# 03 — Tuning and final evaluation (standalone)

Runs from a **fresh Colab session** top to bottom. It rebuilds the data, loads your
trained model from Drive, tunes the confidence threshold on one data group, and
reports the final score on a separate group — the fix your mentor asked for.

**It does not retrain.** Your model is loaded from Drive.

First: Runtime → Change runtime type → GPU.


### Step 1 — Install


In [ ]:
!pip install -q ultralytics huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.4 MB/s eta 0:00:00


### Step 2 — Mount Drive (where trained model is saved)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
MODEL_SRC = Path('/content/drive/MyDrive/metro-crowd-detection/weights/finetuned.pt')
print('model found in Drive:', MODEL_SRC.exists())


Mounted at /content/drive
model found in Drive: True


### Step 3 — Loading trained model

Copies the model out of Drive into the session. No training happens.


In [ ]:
import shutil
shutil.copy(MODEL_SRC, '/content/best.pt')
from ultralytics import YOLO
model = YOLO('/content/best.pt')
print('model loaded')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
model loaded


### Step 4 — Rebuild the data

The images were erased when the session closed, so download and convert them
again — exactly as in training (validation set, head boxes). This matches how I
trained (`VAL_ONLY = True`).


In [ ]:
from huggingface_hub import snapshot_download
import zipfile

raw = snapshot_download(repo_id='sshao0516/CrowdHuman', repo_type='dataset',
                        local_dir='/content/chraw',
                        allow_patterns=['*.odgt','CrowdHuman_val*'])
RAW = Path('/content/chraw'); SRC = Path('/content/crowdhuman'); SRC.mkdir(exist_ok=True)
for odgt in RAW.rglob('*.odgt'): shutil.copy2(odgt, SRC/odgt.name)

(SRC/'Images_val').mkdir(parents=True, exist_ok=True)
for z in sorted(RAW.rglob('CrowdHuman_val*.zip')):
    print('unzip', z.name)
    with zipfile.ZipFile(z) as zf: zf.extractall('/content/_tmp')
for p in Path('/content/_tmp').rglob('*'):
    if p.suffix.lower() in {'.jpg','.jpeg','.png'}:
        d = SRC/'Images_val'/p.name
        if not d.exists(): shutil.move(str(p), d)
print('val images:', len(list((SRC/'Images_val').glob('*.jpg'))))


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

unzip CrowdHuman_val.zip
val images: 4370


### Step 5 — Convert to YOLO labels (head boxes), same as training


In [ ]:
import json, math, random
from PIL import Image

def clamp(v,lo=0.,hi=1.): return max(lo,min(hi,v))
def to_line(box,iw,ih):
    x,y,w,h=box
    if w<=0 or h<=0: return None
    x1,y1=max(0,x),max(0,y); x2,y2=min(iw,x+w),min(ih,y+h)
    if x2<=x1 or y2<=y1: return None
    return f'0 {clamp(((x1+x2)/2)/iw):.6f} {clamp(((y1+y2)/2)/ih):.6f} {clamp((x2-x1)/iw):.6f} {clamp((y2-y1)/ih):.6f}'

def convert(odgt, images_dir, out):
    oi=Path(out)/'images'; ol=Path(out)/'labels'
    oi.mkdir(parents=True,exist_ok=True); ol.mkdir(parents=True,exist_ok=True)
    n=0
    for line in open(odgt):
        line=line.strip()
        if not line: continue
        r=json.loads(line); src=None
        for e in ('.jpg','.jpeg','.png'):
            cand=Path(images_dir)/f"{r['ID']}{e}"
            if cand.exists(): src=cand; break
        if src is None: continue
        try:
            with Image.open(src) as im: iw,ih=im.size
        except: continue
        lines=[]
        for gt in r.get('gtboxes',[]):
            if gt.get('tag')!='person': continue
            if gt.get('extra',{}).get('ignore',0) or gt.get('head_attr',{}).get('ignore',0): continue
            b=gt.get('hbox')
            if not b or len(b)!=4: continue
            ln=to_line(b,iw,ih)
            if ln: lines.append(ln)
        (ol/f"{r['ID']}.txt").write_text('\n'.join(lines))
        d=oi/src.name
        if not d.exists():
            try: d.symlink_to(src.resolve())
            except: shutil.copy2(src,d)
        n+=1
    return n

DATA=Path('/content/data')
print('converting...')
convert(SRC/'annotation_val.odgt', SRC/'Images_val', DATA/'all')
print('done:', len(list((DATA/'all'/'images').glob('*'))), 'images')


converting...
done: 4370 images


### Step 6 — Make the same two groups you trained with

I trained with `VAL_ONLY = True`, so the notebook had split the validation images
80/20 into a train part and a test part (seed 0). We reproduce that exact split so
the **tuning** uses the 80% the model learned from, and the **final test** uses the
20% it never saw. Same seed = same split as training.


In [ ]:
imgs = sorted((DATA/'all'/'images').glob('*'))
random.seed(0)          # same seed the training notebook used
random.shuffle(imgs)
cut = int(len(imgs)*0.8)
train_part = imgs[:cut]   # model learned from these -> use for TUNING
test_part  = imgs[cut:]   # model never saw these   -> use for FINAL TEST

def count_error(paths, conf, limit=None):
    if limit: paths=paths[:limit]
    ae,se,n=[],[],0
    for p in paths:
        lab=DATA/'all'/'labels'/(p.stem+'.txt')
        if not lab.exists(): continue
        true_n=sum(1 for l in lab.read_text().splitlines() if l.strip())
        pred_n=len(model.predict(source=str(p),conf=conf,classes=[0],verbose=False)[0].boxes)
        d=pred_n-true_n; ae.append(abs(d)); se.append(d*d); n+=1
    if not n: return None
    return {'images':n,'mae':round(sum(ae)/n,3),'rmse':round(math.sqrt(sum(se)/n),3)}

print('tuning group:', len(train_part), ' final-test group:', len(test_part))


tuning group: 3496  final-test group: 874


### Step 7 — Tuning: choose the confidence on the tuning group

Tries several thresholds on the group the model learned from, and picks the one with
the lowest counting error. This is where the setting is chosen — not on the test.


In [ ]:
CANDIDATES=[0.20,0.25,0.30,0.35,0.40,0.45,0.50]
print('conf   MAE     RMSE   (tuning group)')
res=[]
for cc in CANDIDATES:
    r=count_error(train_part, cc, limit=300)
    res.append((cc,r)); print(f' {cc:.2f}  {r["mae"]:.3f}  {r["rmse"]:.3f}')
best=min(res,key=lambda t:t[1]['mae'])[0]
print('\nbest confidence:', best)


conf   MAE     RMSE   (tuning group)
 0.20  4.247  8.815
 0.25  4.523  8.813
 0.30  4.967  9.515
 0.35  5.480  10.283
 0.40  6.153  11.339
 0.45  6.833  12.351
 0.50  7.600  13.532

best confidence: 0.2


### Step 8 — Final test: measure once on the held-out group

Applies the chosen confidence to the 20% the model never saw. This is clean
final number. Detection metrics come from a small yaml pointing only at this group.


In [ ]:
# build a yaml + folder for the held-out test group only
test_dir=Path('/content/testset'); (test_dir/'images').mkdir(parents=True,exist_ok=True); (test_dir/'labels').mkdir(parents=True,exist_ok=True)
for p in test_part:
    d=test_dir/'images'/p.name
    if not d.exists():
        try: d.symlink_to(p.resolve())
        except: shutil.copy2(p,d)
    lab=DATA/'all'/'labels'/(p.stem+'.txt')
    if lab.exists():
        dl=test_dir/'labels'/lab.name
        if not dl.exists(): shutil.copy2(lab,dl)
Path('/content/testset.yaml').write_text(f'path: {test_dir}\ntrain: images\nval: images\nnames:\n  0: person\n')

mt=model.val(data='/content/testset.yaml', conf=best, verbose=False)
final_count=count_error(test_part, best, limit=None)

final={'confidence_tuned':best,'map50':round(float(mt.box.map50),4),
       'precision':round(float(mt.box.mp),4),'recall':round(float(mt.box.mr),4),
       'count_mae':final_count['mae'],'count_rmse':final_count['rmse'],
       'test_images':final_count['images']}

R=Path('/content/results'); R.mkdir(exist_ok=True)
table=('# Final evaluation (tuned and tested on separate data)\n\n'
       f'Confidence {best} chosen on the tuning group, applied once to the held-out group.\n\n'
       '| Metric | Value |\n|---|---|\n'
       f'| Confidence (tuned) | {best} |\n| mAP@0.5 | {final["map50"]} |\n'
       f'| Precision | {final["precision"]} |\n| Recall | {final["recall"]} |\n'
       f'| Count MAE | {final["count_mae"]} |\n| Count RMSE | {final["count_rmse"]} |\n'
       f'| Test images | {final["test_images"]} |\n')
(R/'final_eval.md').write_text(table)
import json as J; (R/'final_eval.json').write_text(J.dumps(final,indent=2))
# save to Drive too
dst=Path('/content/drive/MyDrive/metro-crowd-detection/results'); dst.mkdir(parents=True,exist_ok=True)
shutil.copy2(R/'final_eval.md',dst/'final_eval.md'); shutil.copy2(R/'final_eval.json',dst/'final_eval.json')
print(table)


Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3486.9±732.3 MB/s, size: 986.3 KB)
val: Scanning /content/testset/labels... 874 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 874/874 2.1Kit/s 0.4s
val: /content/testset/images/273278,6398f000dc4fa731.jpg: 1 duplicate labels removed
val: /content/testset/images/283647,9c57000bbf32069.jpg: 1 duplicate labels removed
val: /content/testset/images/283992,12000e7e7d513.jpg: 1 duplicate labels removed
val: New cache created: /content/testset/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 2.0it/s 27.6s
                   all        874      19314      0.832      0.664      0.677      0.425
Speed: 2.9ms preprocess, 7.0ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/runs/detect/val
# Final evaluation (tuned and tested on separate data)

Confidence

### Summary
> The confidence threshold was selected by comparing counting error across several
> values on the portion of data the model was trained on, then applied once to a
> held-out portion the model never saw, which served as the final test. Tuning and
> final evaluation therefore used separate data.

> Because a subset of CrowdHuman was used (the validation archive, split into train
> and held-out test), the reported figures are a proof-of-concept evaluation rather
> than a full-benchmark result.

